# Molecular phase estimation for LiH

Regenerate the molecular model, estimate its energy and compare with full CI. The quantum circuit and phase marginal are computed locally with FlagQuantum.

Use the local FlagQuantum installation.

In [ ]:
import math
from pathlib import Path

from IPython.display import Image, display

import flagquantum as fq

Import the native workflow and its molecular model.

In [ ]:
from examples.synthesis.applications.chemistry.qpe_for_molecules.overview import (
    model,
    workflow,
)
from examples.synthesis.applications.chemistry.qpe_for_molecules.overview.main import (
    print_report,
    save_figures,
)

print(fq.__name__)

Inspect the candidate geometries and select LiH at 1.596 Å.

In [ ]:
print(model.GEOMETRIES)
print("Selected geometry:", model.GEOMETRIES["LiH"])

Recompute electronic integrals and full and active CI.

In [ ]:
lih = model.build_model()
print("RHF energy:", lih.rhf)

Encode the six-qubit symmetry sector containing the HF determinant.

In [ ]:
print("Sector dimension:", len(model.sector_words()))
print("Hartree–Fock occupation:", model.sector_words()[0])

Keep both exact energy references.

In [ ]:
print("Full CI and active CI:", lih.full_ci, lih.active_ci)

Inspect the freshly tapered coefficient spectrum.

In [ ]:
print("Largest and smallest retained numerical magnitudes:", lih.coefficient_magnitudes[0], lih.coefficient_magnitudes[-1])

Apply the 0.03 Ha coefficient cutoff.

In [ ]:
print("Cutoff and retained terms:", model.CUTOFF, len(lih.terms))

Recover the scale and the removed scalar energy.

In [ ]:
print("Normalization and offset:", lih.normalization, lih.offset)

Compute the ordered second-order commutator bound.

In [ ]:
gamma = model.gamma_two(lih.terms)
print("Gamma and target error:", gamma, workflow.EPS)

Use the original seven phase bits.

In [ ]:
print("Powers:", workflow.POWERS)

Compute the adaptive theoretical schedule.

In [ ]:
r0 = math.sqrt(2**5 * gamma / (3 * workflow.EPS)) * (2 * math.pi) ** 1.5
print("Adaptive bound:", tuple(math.ceil(r0 * p**1.5) for p in workflow.POWERS))

Compare with naïve powers of the first approximation.

In [ ]:
print("Naive bound:", tuple(math.ceil(r0 * p) for p in workflow.POWERS))

Use the experimental per-power step counts in the quantum program.

In [ ]:
print("Experimental repetitions:", workflow.STEPS)

Prepare the Hartree–Fock state in the tapered representation.

In [ ]:
print("Tapered Hartree–Fock word:", model.sector_words().index(model.HF_OCCUPATION))

Build the native controlled product formulas and inverse QFT.

In [ ]:
program = workflow.build_program(lih)
print("Compiled phase register:", len(program.registers["phase"]))

Execute locally and draw a seeded finite sample of 1,000 outcomes.

In [ ]:
result = workflow.read_run(lih, program)
print("Total draws:", sum(result.counts))

Display the coefficient spectrum and the energy histogram.

In [ ]:
figures = Path("notebook-figures")
save_figures(result, figures)
display(Image(filename=str(figures / "coefficient-magnitudes.png")))
display(Image(filename=str(figures / "energy-histogram.png")))

Read the most probable energy and the lowest detected peak alongside the full CI reference.

In [ ]:
print_report(result)